In [ ]:
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

In [ ]:
import matplotlib.pyplot as plt

plt.rc('font', family='NanumBarunGothic')
plt.plot([1, 2, 3], [1, 4, 9])
plt.title('맑은 고딕 테스트')
plt.show()

In [ ]:
# 드라이브에서 파일 가져오기
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np

# 파일 경로
img_file_path = '/content/drive/My Drive/apple_images.npy'
label_file_path = '/content/drive/My Drive/apple_labels.npy'

# 파일 로드
img_data = np.load(img_file_path)
label_data = np.load(label_file_path)

print(img_data.shape)
print(label_data.shape)

In [ ]:
# 0: 특, 1: 상, 2: 보통
from typing import Counter
count = Counter(label_data)
print(count)

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical

# 이미지 데이터 정규화 및 타입 변환
img_data = np.array(img_data, dtype=np.float32) / 255.0
label_data = np.array(label_data, dtype=int)

# 데이터 나누기
x_train, x_test, y_train, y_test = train_test_split(img_data, label_data, test_size=0.1, stratify=label_data, random_state=42)

# One-Hot Encoding
y_train_categorical = to_categorical(y_train, num_classes=3)
y_test_categorical = to_categorical(y_test, num_classes=3)

# CNN 모델 설계
# 다음은 모델에 필요한 모듈을 불러온 후, 모델 층을 쌓는 코드 입니다.
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.models import Sequential

# 모델 생성 (순차 모델)
model = Sequential()

# 첫 번째 합성곱 층: 16개의 필터, 크기 (3, 3), 활성화 함수 ReLU, 입력 크기 (224, 224, 3)
model.add(Conv2D(16, (3, 3), activation='relu', input_shape=(224, 224, 3)))
model.add(BatchNormalization())   # 배치 정규화 추가 (출력의 분포를 안정화시킴)
model.add(MaxPooling2D((2, 2)))   # 풀링 층: (2, 2) 크기의 맥스 풀링

# 두 번째 합성곱 층: 32개의 필터, 크기 (3, 3), 활성화 함수 ReLU
model.add(Conv2D(32, (3, 3), activation='relu'))
model.add(BatchNormalization())  # 배치 정규화 추가
model.add(MaxPooling2D((2, 2)))  # 풀링 층: (2, 2) 크기의 맥스 풀링

# 세 번째 합성곱 층: 64개의 필터, 크기 (3, 3), 활성화 함수 ReLU
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(BatchNormalization())  # 배치 정규화 추가
model.add(MaxPooling2D((2, 2)))  # 풀링 층: (2, 2) 크기의 맥스 풀링

# GlobalAveragePooling2D는 Flatten보다 과적합에 덜 민감함
model.add(GlobalAveragePooling2D())

# 드롭아웃 층: 50%의 뉴런을 무작위로 비활성화하여 과적합을 방지
model.add(Dropout(0.5))

# 첫 번째 완전 연결 층: 128개의 뉴런, 활성화 함수 ReLU
model.add(Dense(256, activation='relu'))

# 드롭아웃 층: 50%의 뉴런을 무작위로 비활성화하여 과적합을 방지
model.add(Dropout(0.5))

# 두 번째 완전 연결 층
# 클래스가 특, 상, 보통 3개이므로 뉴런 3개, softmax 활성화 함수를 사용하였습니다.
model.add(Dense(3, activation='softmax'))

model.summary()

In [30]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.metrics import Recall, Precision, AUC

optimizer = Adam(learning_rate=0.0001)

# 다중 클래스 분류를 위한 컴파일 설정
model.compile(
   optimizer=optimizer,
   loss='categorical_crossentropy',  # 손실함수
   metrics=[
      'accuracy',
      Recall(name='recall'),
      Precision(name='precision'),
      AUC(name='auc')
   ]
)

# ModelCheckpoint 콜백 설정
model_checkpoint = ModelCheckpoint('best_model.keras', monitor='val_auc', save_best_only=True, mode='max', verbose=1)

In [31]:
# 모델 학습
base_history = model.fit(x_train, y_train_categorical, batch_size=16, validation_split=0.2, epochs=10, callbacks=[model_checkpoint])

Epoch 1/10
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - accuracy: 0.3177 - auc: 0.4906 - loss: 1.0989 - precision: 0.0000e+00 - recall: 0.0000e+00
Epoch 1: val_auc improved from -inf to 0.53723, saving model to best_model.keras
71/71 ━━━━━━━━━━━━━━━━━━━━ 24s 224ms/step - accuracy: 0.3179 - auc: 0.4906 - loss: 1.0989 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.3830 - val_auc: 0.5372 - val_loss: 1.0975 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/10
69/71 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.3577 - auc: 0.5374 - loss: 1.0968 - precision: 0.0000e+00 - recall: 0.0000e+00
Epoch 2: val_auc did not improve from 0.53723
71/71 ━━━━━━━━━━━━━━━━━━━━ 23s 38ms/step - accuracy: 0.3584 - auc: 0.5391 - loss: 1.0965 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.3830 - val_auc: 0.5319 - val_loss: 1.1470 - val_precision: 0.3830 - val_recall: 0.3830
Epoch 3/10
70/71 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.5372 - auc: 0.7144 - loss: 1.019

In [ ]:
# 모델 평가 (여러 지표가 반환되는 경우)
eval_results = model.evaluate(x_test, y_test_categorical)

# 각 평가 지표 출력
print(f"Test Loss: {eval_results[0]}")
print(f"Test Accuracy: {eval_results[1]}")
print(f"AUC: {eval_results[2]}")
print(f"Precision: {eval_results[3]}")
print(f"Recall: {eval_results[4]}")

In [ ]:
from sklearn.metrics import accuracy_score

# 모델 예측
y_pred = model.predict(x_test)
y_pred_arg = np.argmax(y_pred, axis=1)

y_pred_arg